# AliNet - Entity Alignment on DBP15K

> **Knowledge Graph Alignment Network with Gated Multi-hop Neighborhood Aggregation**
> Zequn Sun, Chengming Wang, Wei Hu, Muhao Chen, Jian Dai, Wei Zhang, Yuzhong Qu (*AAAI 2020*)
> Source: https://arxiv.org/pdf/1911.08936v1

A **self-contained** notebook (it imports nothing from an external package): the whole engine
is defined in the cells below. The same implementation is available in the `code/` package
(`code/src/models/alinet.py`, class `AliNetTrainer` in `code/src/trainer.py`).

## The AliNet idea
The 1-hop neighbourhoods of two aligned entities are often **non-isomorphic** from one KG to
the other. AliNet mitigates this with a **gated multi-hop GNN**:

1. **1-hop aggregation**: a GCN pass over the symmetrically-normalised adjacency `A_hat` (with
   self-loops): `g1 = A_hat (H W1)`.
2. **2-hop aggregation with attention**: distant neighbours are noisy, so they are aggregated
   by attention (GAT) over the (sampled/capped) 2-hop edges.
3. **Gate**: combines the two: `out = sigmoid(g1 W_g).g1 + (1-sigmoid).g2`.

The final representation is the **concatenation (JK)** of the layer outputs, L2-normalised.

## Key ingredients found to match the paper (see the Notes section)
- **Linear propagation** (no ReLU between layers): a ReLU destroys the alignment signal.
- **Unit init** of the entity embeddings + JK **without** the raw embedding (else memorisation).
- **Relation-aware (TransE) loss** `||z_h + r - z_t||`: a **structural anchor for ALL entities**
  (not just the 11% seeds) - the decisive link.
- **Mixed hard (epsilon-truncated) negatives**: once the geometry is anchored, they refine Hit@1.

## Results (DBP15K zh_en, 30% seed)
| | Hit@1 | Hit@5 | Hit@10 | MRR |
|---|---:|---:|---:|---:|
| **This notebook** | ~0.53 | ~0.76 | ~0.81 | ~0.63 |
| AliNet (paper) | 0.539 | - | 0.826 | 0.628 |

## Metrics: **MRR, Hit@1, Hit@5, Hit@10** (+ CSLS).

---
## 1. Environment, imports and style

In [ ]:
import os, sys, csv, time, random, logging
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from collections import Counter

import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from tqdm import tqdm          # plain terminal tqdm (not the notebook widget)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root :", PROJECT_ROOT)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

SEED = 2024
set_seed(SEED)
torch.backends.cudnn.benchmark = True

### 1.1 Modern dark theme (identical to `code/src/utils/plotting.py`)

In [ ]:
"""Modern dark-theme plotting helpers.

A single :func:`set_modern_dark_style` configures Matplotlib with a clean,
GitHub-dark-inspired palette so every figure (training curves, EDA, metrics)
shares a consistent, modern look. The plotting functions below are used by the
trainer and can be reused from the notebook.
"""


# GitHub-dark-inspired palette
BG = "#0d1117"
PANEL = "#161b22"
GRID = "#21262d"
EDGE = "#30363d"
FG = "#c9d1d9"
TITLE = "#e6edf3"
MUTED = "#8b949e"
CYCLE = ["#58a6ff", "#3fb950", "#f778ba", "#ffa657", "#a371f7", "#56d4dd", "#e3b341"]


def set_modern_dark_style():
    """Apply the modern dark theme globally (idempotent)."""
    mpl.rcParams.update({
        "figure.facecolor": BG,
        "figure.edgecolor": BG,
        "savefig.facecolor": BG,
        "savefig.edgecolor": BG,
        "axes.facecolor": PANEL,
        "axes.edgecolor": EDGE,
        "axes.labelcolor": FG,
        "axes.titlecolor": TITLE,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.linewidth": 1.0,
        "grid.color": GRID,
        "grid.linestyle": "--",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "text.color": FG,
        "legend.facecolor": PANEL,
        "legend.edgecolor": EDGE,
        "legend.framealpha": 0.9,
        "lines.linewidth": 2.2,
        "lines.markersize": 5,
        "lines.solid_capstyle": "round",
        "font.size": 11,
        "figure.dpi": 120,
        "axes.prop_cycle": mpl.cycler(color=CYCLE),
    })


def style_axes(ax, title=None, xlabel=None, ylabel=None):
    """Apply consistent spine/tick styling to an Axes."""
    if title:
        ax.set_title(title, pad=12)
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(length=0)
    return ax


def plot_loss_curves(loss_hist, ax=None, keys=("loss", "kge", "align", "pseudo")):
    """Plot per-epoch loss components from a list of dicts (with 'epoch')."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ep = [h["epoch"] for h in loss_hist]
    labels = {"loss": "total", "kge": "kge (TransE)",
              "align": "align (seed)", "pseudo": "align (pseudo)"}
    for key in keys:
        ax.plot(ep, [h.get(key, 0.0) for h in loss_hist], label=labels.get(key, key))
    ax.legend()
    return style_axes(ax, "Training loss", "epoch", "loss")


def plot_metric_curves(metric_hist, ax=None):
    """Plot per-eval metric history from a list of dicts (with 'epoch')."""
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 5))
    ep = [h["epoch"] for h in metric_hist]
    for key in [k for k in metric_hist[0] if k != "epoch"]:
        ax.plot(ep, [h[key] for h in metric_hist], marker="o", label=key)
    ax.set_ylim(0, 1)
    ax.legend()
    return style_axes(ax, "Test metrics (avg direction)", "epoch", "score")

set_modern_dark_style()
print('Dark theme applique.')

---
## 2. Configuration (YAML) and logging
Driven by **`configs/alinet_dbp15k.yaml`**. Code identical to `code/src/utils/config.py` and `logger.py`.

In [ ]:
"""Configuration loading and run-directory bootstrap.

The whole experiment is driven by a single YAML file (see ``configs/``).
This module:
  * loads that YAML into a dotted-access namespace (``cfg.train.lr`` etc.),
  * creates a timestamped *run directory* under ``logging.output_dir``,
  * dumps the resolved config back to disk for reproducibility.

The logger is created separately by :mod:`src.utils.logger`.
"""




class Cfg(dict):
    """A ``dict`` that also supports attribute access, recursively.

    ``cfg.train.lr`` is equivalent to ``cfg["train"]["lr"]``. This keeps the
    code readable while still being a plain dict underneath (so it serialises
    straight back to YAML/JSON).
    """

    def __init__(self, d: dict | None = None):
        super().__init__()
        for k, v in (d or {}).items():
            self[k] = Cfg(v) if isinstance(v, dict) else v

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError as e:
            raise AttributeError(name) from e

    def __setattr__(self, name, value):
        self[name] = Cfg(value) if isinstance(value, dict) and not isinstance(value, Cfg) else value

    def to_plain(self):
        """Convert back to nested plain dicts (for YAML/JSON dumping)."""
        return {k: (v.to_plain() if isinstance(v, Cfg) else v) for k, v in self.items()}


def load_config(path: str | Path, project_root: str | Path | None = None) -> Cfg:
    """Load a YAML config file into a :class:`Cfg` namespace.

    ``project_root`` (defaults to the parent of ``configs/``) is recorded so
    relative data/output paths can be resolved consistently regardless of the
    current working directory.
    """
    path = Path(path).resolve()
    with open(path, "r", encoding="utf-8") as f:
        raw = yaml.safe_load(f)
    cfg = Cfg(raw)
    root = Path(project_root).resolve() if project_root else path.parent.parent
    cfg._project_root = str(root)
    cfg._config_path = str(path)
    return cfg


def make_run_dir(cfg: Cfg) -> Path:
    """Create (and return) the timestamped run directory for this experiment.

    Layout::

        <output_dir>/<experiment.name>_<YYYYmmdd-HHMMSS>/
            training.txt          (full training log)
            config_used.yaml      (snapshot of the resolved config)
            model.pt / model_best.pt
            embeddings.pt
            metrics.csv / loss.csv
            *.png                 (loss & metric curves)
    """
    root = Path(cfg._project_root)
    out = root / cfg.logging.output_dir
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = out / f"{cfg.experiment.name}_{stamp}"
    run_dir.mkdir(parents=True, exist_ok=True)
    cfg._run_dir = str(run_dir)

    # snapshot the exact config used for this run
    with open(run_dir / cfg.logging.config_dump, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg.to_plain(), f, sort_keys=False, allow_unicode=True)
    return run_dir

In [ ]:
"""Logging setup: write simultaneously to the console and to ``training.txt``."""




def get_logger(cfg: Cfg, run_dir: Path) -> logging.Logger:
    """Return a logger writing to both stdout and ``training.txt``.

    Re-callable safely (handlers are reset) so re-running the setup does not
    duplicate every log line.
    """
    logger = logging.getLogger(cfg.experiment.name)
    logger.setLevel(getattr(logging, cfg.logging.log_level.upper(), logging.INFO))
    logger.handlers.clear()
    logger.propagate = False

    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")

    fh = logging.FileHandler(run_dir / cfg.logging.log_file, mode="w", encoding="utf-8")
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    return logger

In [ ]:
cfg = load_config(PROJECT_ROOT / "configs" / "alinet_dbp15k.yaml", project_root=PROJECT_ROOT)
cfg.experiment.seed = SEED
cfg.experiment.name = f"alinet_{cfg.data.lang}_{cfg.data.fold}"

run_dir = make_run_dir(cfg)
logger = get_logger(cfg, run_dir)
device = torch.device(cfg.experiment.device if torch.cuda.is_available() else "cpu")
logger.info(f"Run dir : {run_dir}")
logger.info(f"Device  : {device}")
print(yaml.safe_dump(cfg.to_plain(), sort_keys=False, allow_unicode=True))

---
## 3. DBP15K data and graph construction

JAPE/MTransE format (disjoint contiguous ids). Code identical to `code/src/data.py` (includes
AliNet's graph builder: normalised 1-hop adjacency + capped 2-hop edges).

In [ ]:
"""DBP15K data loading + neighbourhood construction for NAEA.

Data layout (the clean JAPE / MTransE split we use, ``<lang>/mtranse/<fold>/``):

    ent_ids_1 / ent_ids_2 : "<id>\\t<uri>"   entities of KG1 (e.g. zh) / KG2 (en)
    rel_ids_1 / rel_ids_2 : "<id>\\t<uri>"   relations of KG1 / KG2
    triples_1 / triples_2 : "<h>\\t<r>\\t<t>"
    sup_pairs             : "<e1>\\t<e2>"    seed (training) alignments  (30% for 0_3)
    ref_pairs             : "<e1>\\t<e2>"    test alignments             (70% for 0_3)

Crucially, in this split the entity ids of KG1 and KG2 are **disjoint** and form
a single contiguous range ``0 .. num_entities-1``; likewise relation ids. That
lets us use one shared embedding table and index it directly.
"""




# --------------------------------------------------------------------------- #
#  Raw file readers
# --------------------------------------------------------------------------- #
def _read_id_uri(path: Path) -> dict[int, str]:
    d = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 2:
                d[int(parts[0])] = parts[1]
    return d


def _read_triples(path: Path) -> np.ndarray:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) >= 3:
                rows.append((int(p[0]), int(p[1]), int(p[2])))
    return np.asarray(rows, dtype=np.int64)


def _read_pairs(path: Path) -> np.ndarray:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.rstrip("\n").split("\t")
            if len(p) >= 2:
                rows.append((int(p[0]), int(p[1])))
    return np.asarray(rows, dtype=np.int64)


# --------------------------------------------------------------------------- #
#  Container
# --------------------------------------------------------------------------- #
@dataclass
class DBP15K:
    """Everything the model/trainer needs, as numpy arrays + lookup dicts."""

    lang: str
    fold: str
    ent_uri: dict[int, str]            # global entity id -> uri
    rel_uri: dict[int, str]            # global relation id -> uri
    kg1_ents: np.ndarray               # entity ids belonging to KG1
    kg2_ents: np.ndarray               # entity ids belonging to KG2
    triples1: np.ndarray               # (M1, 3)  KG1 triples
    triples2: np.ndarray               # (M2, 3)  KG2 triples
    train_pairs: np.ndarray            # (S, 2)   seed alignments
    test_pairs: np.ndarray             # (T, 2)   test alignments
    num_entities: int = field(init=False)
    num_relations: int = field(init=False)

    def __post_init__(self):
        self.num_entities = int(max(self.ent_uri) + 1)
        self.num_relations = int(max(self.rel_uri) + 1)

    @property
    def triples(self) -> np.ndarray:
        return np.concatenate([self.triples1, self.triples2], axis=0)

    def summary(self) -> dict:
        return {
            "lang": self.lang,
            "fold": self.fold,
            "num_entities": self.num_entities,
            "num_relations": self.num_relations,
            "|KG1 ents|": len(self.kg1_ents),
            "|KG2 ents|": len(self.kg2_ents),
            "|KG1 triples|": len(self.triples1),
            "|KG2 triples|": len(self.triples2),
            "train_pairs": len(self.train_pairs),
            "test_pairs": len(self.test_pairs),
        }


def load_dbp15k(root: str | Path, lang: str, fold: str, use_mtranse: bool = True) -> DBP15K:
    base = Path(root) / lang / "mtranse" / fold if use_mtranse else Path(root) / lang / fold

    ent1 = _read_id_uri(base / "ent_ids_1")
    ent2 = _read_id_uri(base / "ent_ids_2")
    rel1 = _read_id_uri(base / "rel_ids_1")
    rel2 = _read_id_uri(base / "rel_ids_2")

    ent_uri = {**ent1, **ent2}
    rel_uri = {**rel1, **rel2}

    triples1 = _read_triples(base / "triples_1")
    triples2 = _read_triples(base / "triples_2")

    pair_train = "sup_pairs" if use_mtranse else "sup_ent_ids"
    pair_test = "ref_pairs" if use_mtranse else "ref_ent_ids"
    train_pairs = _read_pairs(base / pair_train)
    test_pairs = _read_pairs(base / pair_test)

    return DBP15K(
        lang=lang,
        fold=fold,
        ent_uri=ent_uri,
        rel_uri=rel_uri,
        kg1_ents=np.asarray(sorted(ent1), dtype=np.int64),
        kg2_ents=np.asarray(sorted(ent2), dtype=np.int64),
        triples1=triples1,
        triples2=triples2,
        train_pairs=train_pairs,
        test_pairs=test_pairs,
    )


# --------------------------------------------------------------------------- #
#  Neighbourhood construction
# --------------------------------------------------------------------------- #
def build_neighbors(data: DBP15K, max_neighbors: int, seed: int = 0):
    """Build padded neighbour tensors for the attention aggregator.

    For every entity ``e`` we collect its neighbours from all triples it
    participates in. A neighbour is a *(relation, entity, direction)* triplet:

      * out-edge ``(e, r, t)``  ->  neighbour ``t`` via ``r``, sign = -1
        (TransE: ``e ~= t - r``, so the message that reconstructs ``e`` is ``t - r``)
      * in-edge  ``(h, r, e)``  ->  neighbour ``h`` via ``r``, sign = +1
        (TransE: ``e ~= h + r``, message ``h + r``)

    Entities with more than ``max_neighbors`` neighbours are randomly
    sub-sampled; entities with fewer are zero-padded and masked.

    Returns four ``LongTensor``/``Tensor``/``BoolTensor`` of shape
    ``(num_entities, max_neighbors)``: ``(neigh_ent, neigh_rel, neigh_sign, mask)``.
    """
    rng = random.Random(seed)
    N = data.num_entities
    adj: list[list[tuple[int, int, int]]] = [[] for _ in range(N)]

    for arr in (data.triples1, data.triples2):
        for h, r, t in arr:
            adj[h].append((int(r), int(t), -1))   # out-edge:  e=h, message t - r
            adj[t].append((int(r), int(h), +1))   # in-edge:   e=t, message h + r

    K = max_neighbors
    neigh_ent = np.zeros((N, K), dtype=np.int64)
    neigh_rel = np.zeros((N, K), dtype=np.int64)
    neigh_sign = np.zeros((N, K), dtype=np.float32)
    mask = np.zeros((N, K), dtype=bool)

    for e in range(N):
        nbrs = adj[e]
        if len(nbrs) > K:
            nbrs = rng.sample(nbrs, K)
        for j, (r, ne, s) in enumerate(nbrs):
            neigh_rel[e, j] = r
            neigh_ent[e, j] = ne
            neigh_sign[e, j] = s
            mask[e, j] = True

    return (
        torch.from_numpy(neigh_ent),
        torch.from_numpy(neigh_rel),
        torch.from_numpy(neigh_sign),
        torch.from_numpy(mask),
    )


# --------------------------------------------------------------------------- #
#  Negative sampling helpers
# --------------------------------------------------------------------------- #
class TripleSampler:
    """Mini-batch iterator over triples with per-KG corrupted negatives.

    Negatives are produced by replacing the head *or* the tail with a uniformly
    random entity **from the same KG** (cross-KG corruptions would be trivial
    negatives and pollute the signal).
    """

    def __init__(self, data: DBP15K, device, neg: int = 5):
        self.device = device
        self.neg = neg
        t1 = torch.from_numpy(data.triples1)
        t2 = torch.from_numpy(data.triples2)
        kg = torch.cat([torch.zeros(len(t1), dtype=torch.long),
                        torch.ones(len(t2), dtype=torch.long)])
        self.triples = torch.cat([t1, t2], 0).to(device)        # (M,3)
        self.kg = kg.to(device)                                  # (M,)
        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(device)

    def __len__(self):
        return len(self.triples)

    def _rand_same_kg(self, kg_flags: torch.Tensor) -> torch.Tensor:
        """A random entity from the same KG as each row of ``kg_flags``."""
        n = kg_flags.shape[0]
        r1 = self.kg1_ents[torch.randint(len(self.kg1_ents), (n,), device=self.device)]
        r2 = self.kg2_ents[torch.randint(len(self.kg2_ents), (n,), device=self.device)]
        return torch.where(kg_flags == 0, r1, r2)

    def batches(self, batch_size: int, shuffle: bool = True):
        M = len(self.triples)
        order = torch.randperm(M, device=self.device) if shuffle else torch.arange(M, device=self.device)
        for s in range(0, M, batch_size):
            idx = order[s:s + batch_size]
            pos = self.triples[idx]                              # (B,3)
            kg = self.kg[idx]                                    # (B,)
            B = pos.shape[0]
            # repeat for `neg` corruptions
            pos_r = pos.repeat(self.neg, 1)                      # (B*neg,3)
            kg_r = kg.repeat(self.neg)
            neg = pos_r.clone()
            corrupt_head = torch.rand(B * self.neg, device=self.device) < 0.5
            rnd = self._rand_same_kg(kg_r)
            neg[:, 0] = torch.where(corrupt_head, rnd, neg[:, 0])
            neg[:, 2] = torch.where(~corrupt_head, rnd, neg[:, 2])
            yield pos, neg, kg


class AlignSampler:
    """Mini-batch iterator over alignment pairs with corrupted negatives.

    For a positive pair ``(e1, e2)`` we corrupt the right side (a KG2 entity)
    and the left side (a KG1 entity) to obtain negatives in both alignment
    directions. ``set_pairs`` lets the trainer inject bootstrapped pseudo-pairs.

    **Hard (nearest-neighbour) negatives.** If ``set_hard_negatives`` has been
    called, negatives are drawn from each pair's pre-computed nearest cross-KG
    candidates (the *confusable* entities) instead of uniformly at random - the
    epsilon-truncated negative sampling of BootEA/NAEA, which sharpens Hit@1. Any
    candidate that equals the gold target is replaced by a random one.
    """

    def __init__(self, data: DBP15K, device, neg: int = 5):
        self.device = device
        self.neg = neg
        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(device)
        self.hard_r = None          # (S, C) nearest KG2 candidates per pair (corrupt right)
        self.hard_l = None          # (S, C) nearest KG1 candidates per pair (corrupt left)
        self.set_pairs(data.train_pairs)

    def set_pairs(self, pairs):
        if isinstance(pairs, np.ndarray):
            pairs = torch.from_numpy(pairs)
        self.pairs = pairs.to(self.device).long()
        self.hard_r = self.hard_l = None    # invalidate stale hard-negative tables

    def set_hard_negatives(self, hard_r, hard_l):
        """``hard_r``/``hard_l``: (S, C) LongTensors aligned with ``self.pairs``."""
        self.hard_r = hard_r.to(self.device)
        self.hard_l = hard_l.to(self.device)

    def __len__(self):
        return len(self.pairs)

    def _sample_hard(self, table, rows, n, gold):
        """Pick ``n`` candidates per row from ``table[rows]``, avoiding ``gold``."""
        cand = table[rows]                                       # (B, C)
        C = cand.shape[1]
        sel = torch.randint(C, (cand.shape[0], n), device=self.device)
        out = torch.gather(cand, 1, sel).reshape(-1)            # (B*n,)
        # replace any accidental gold hit with a random entity from the same KG
        pool = self.kg2_ents if table is self.hard_r else self.kg1_ents
        clash = out == gold
        if clash.any():
            out[clash] = pool[torch.randint(len(pool), (int(clash.sum()),), device=self.device)]
        return out

    def batches(self, batch_size: int, shuffle: bool = True):
        S = len(self.pairs)
        order = torch.randperm(S, device=self.device) if shuffle else torch.arange(S, device=self.device)
        for s in range(0, S, batch_size):
            idx = order[s:s + batch_size]
            pos = self.pairs[idx]                                # (B,2)
            B = pos.shape[0]
            n = self.neg
            e1 = pos[:, 0].repeat_interleave(n)
            e2 = pos[:, 1].repeat_interleave(n)
            if self.hard_r is not None and self.hard_l is not None:
                neg_r = self._sample_hard(self.hard_r, idx, n, e2)
                neg_l = self._sample_hard(self.hard_l, idx, n, e1)
            else:
                neg_r = self.kg2_ents[torch.randint(len(self.kg2_ents), (B * n,), device=self.device)]
                neg_l = self.kg1_ents[torch.randint(len(self.kg1_ents), (B * n,), device=self.device)]
            yield pos, (e1, e2, neg_l, neg_r)


# =========================================================================== #
#  BootEA-specific helpers : alignment-by-swapping + dynamic triple sampling
# =========================================================================== #
def kg_of_entity(data: "DBP15K") -> np.ndarray:
    """Return an array ``kg[e] in {0, 1}`` giving the KG each entity belongs to."""
    kg = np.zeros(data.num_entities, dtype=np.int64)
    kg[data.kg2_ents] = 1
    return kg


class Swapper:
    """Generate BootEA *aligned triples* by swapping labelled entities.

    For a labelled pair ``(a, b)`` we substitute ``b`` for ``a`` in each of
    ``a``'s triples and vice-versa, e.g. ``(a, r, t) -> (b, r, t)``. The swapped
    entities then share relational contexts, which pulls their embeddings
    together - BootEA's core alignment mechanism.

    Per-entity triple lists are precomputed once; :meth:`generate` is then cheap
    enough to be re-run every bootstrapping round on the (growing) labelled set.
    """

    def __init__(self, data: "DBP15K"):
        triples = np.concatenate([data.triples1, data.triples2], axis=0)
        self.triples = triples
        n = data.num_entities
        self.as_head = [[] for _ in range(n)]
        self.as_tail = [[] for _ in range(n)]
        for i, (h, _r, t) in enumerate(triples):
            self.as_head[int(h)].append(i)
            self.as_tail[int(t)].append(i)

    def generate(self, labeled_pairs, cap_per_role: int = 100) -> np.ndarray:
        """Return an ``(K, 3)`` array of swapped (h, r, t) triples for the pairs."""
        T = self.triples
        out = []
        for a, b in labeled_pairs:
            a, b = int(a), int(b)
            for src, dst in ((a, b), (b, a)):          # dst takes src's place
                for ti in self.as_head[src][:cap_per_role]:
                    out.append((dst, T[ti, 1], T[ti, 2]))
                for ti in self.as_tail[src][:cap_per_role]:
                    out.append((T[ti, 0], T[ti, 1], dst))
        if not out:
            return np.empty((0, 3), dtype=np.int64)
        return np.asarray(out, dtype=np.int64)


class DynamicTripleSampler:
    """Mini-batch sampler over a *mutable* set of triples (BootEA / AlignE).

    Unlike :class:`TripleSampler` (which fixes the per-KG split), this sampler
    determines each corrupted entity's KG from a global ``kg_of_ent`` array, so
    it correctly handles **swapped** triples whose head and tail may live in
    different KGs. Negatives are uniform within the corrupted position's KG, or
    **epsilon-truncated** (drawn from that entity's nearest same-KG neighbours) when a
    candidate table has been provided via :meth:`set_candidates`.
    """

    def __init__(self, device, kg_of_ent: np.ndarray, kg1_ents, kg2_ents, neg: int = 5):
        self.device = device
        self.neg = neg
        self.kg_of_ent = torch.as_tensor(kg_of_ent, device=device).long()
        self.kg1_ents = torch.as_tensor(kg1_ents, device=device).long()
        self.kg2_ents = torch.as_tensor(kg2_ents, device=device).long()
        self.cand = None            # (N, C) nearest same-KG neighbours, optional
        self.triples = torch.empty((0, 3), dtype=torch.long, device=device)

    def set_triples(self, triples):
        if isinstance(triples, np.ndarray):
            triples = torch.from_numpy(triples)
        self.triples = triples.to(self.device).long()

    def set_candidates(self, cand):
        """``cand`` : (N, C) LongTensor of nearest same-KG neighbours per entity."""
        self.cand = cand.to(self.device) if cand is not None else None

    def __len__(self):
        return len(self.triples)

    def _rand_same_kg(self, ent_ids):
        kg = self.kg_of_ent[ent_ids]
        n = ent_ids.shape[0]
        r1 = self.kg1_ents[torch.randint(len(self.kg1_ents), (n,), device=self.device)]
        r2 = self.kg2_ents[torch.randint(len(self.kg2_ents), (n,), device=self.device)]
        return torch.where(kg == 0, r1, r2)

    def _trunc_same_kg(self, ent_ids):
        """epsilon-truncated: a random one of each entity's nearest same-KG neighbours."""
        cand = self.cand[ent_ids]                              # (n, C)
        sel = torch.randint(cand.shape[1], (cand.shape[0], 1), device=self.device)
        return torch.gather(cand, 1, sel).squeeze(1)

    def batches(self, batch_size: int, shuffle: bool = True):
        M = len(self.triples)
        order = torch.randperm(M, device=self.device) if shuffle else torch.arange(M, device=self.device)
        for s in range(0, M, batch_size):
            idx = order[s:s + batch_size]
            pos = self.triples[idx]                            # (B,3)
            B = pos.shape[0]
            n = self.neg
            pos_r = pos.repeat(n, 1)                           # (B*n,3)
            neg = pos_r.clone()
            corrupt_head = torch.rand(B * n, device=self.device) < 0.5
            # entity currently occupying the position we corrupt
            orig = torch.where(corrupt_head, pos_r[:, 0], pos_r[:, 2])
            repl = self._trunc_same_kg(orig) if self.cand is not None else self._rand_same_kg(orig)
            neg[:, 0] = torch.where(corrupt_head, repl, neg[:, 0])
            neg[:, 2] = torch.where(~corrupt_head, repl, neg[:, 2])
            yield pos, neg


# =========================================================================== #
#  AliNet-specific helpers : 1-hop normalised adjacency + capped 2-hop edges
# =========================================================================== #
def build_alinet_graph(data: "DBP15K", max_two_hop: int = 10, seed: int = 0):
    """Build the graph structures AliNet needs (relation types ignored).

    Returns
    -------
    adj1 : torch.sparse_coo_tensor (N, N)
        Symmetrically normalised adjacency with self-loops,
        ``A_hat = D^{-1/2} (A + I) D^{-1/2}`` - used for 1-hop GCN aggregation.
    two_hop : LongTensor (2, E)
        Edges ``[dst, src]`` where ``src`` is a (sampled, capped) 2-hop neighbour
        of ``dst`` (excluding 1-hop neighbours and self) - used for the
        attention-based 2-hop aggregation.
    """
    rng = random.Random(seed)
    N = data.num_entities
    nbrs = [set() for _ in range(N)]
    for arr in (data.triples1, data.triples2):
        for h, _r, t in arr:
            h, t = int(h), int(t)
            if h != t:
                nbrs[h].add(t)
                nbrs[t].add(h)

    # ---- 1-hop normalised adjacency (with self-loops) ---------------------
    rows, cols = [], []
    for i in range(N):
        rows.append(i); cols.append(i)            # self-loop
        for j in nbrs[i]:
            rows.append(i); cols.append(j)
    rows = np.asarray(rows, dtype=np.int64)
    cols = np.asarray(cols, dtype=np.int64)
    deg = np.asarray([len(nbrs[i]) + 1 for i in range(N)], dtype=np.float64)  # +1 self-loop
    inv_sqrt = 1.0 / np.sqrt(deg)
    vals = (inv_sqrt[rows] * inv_sqrt[cols]).astype(np.float32)
    adj1 = torch.sparse_coo_tensor(
        torch.from_numpy(np.stack([rows, cols])), torch.from_numpy(vals), (N, N)
    ).coalesce()

    # ---- capped 2-hop edges ----------------------------------------------
    dst, src = [], []
    for i in range(N):
        one = nbrs[i]
        if not one:
            continue
        two = set()
        # bound exploration on hub nodes
        for j in list(one)[:64]:
            two.update(list(nbrs[j])[:64])
        two.discard(i)
        two -= one
        if not two:
            continue
        two = list(two)
        if len(two) > max_two_hop:
            two = rng.sample(two, max_two_hop)
        for k in two:
            dst.append(i); src.append(k)
    two_hop = torch.tensor([dst, src], dtype=torch.long) if dst else torch.zeros((2, 0), dtype=torch.long)
    return adj1, two_hop

In [ ]:
data = load_dbp15k(PROJECT_ROOT / cfg.data.root, cfg.data.lang,
                   cfg.data.fold, cfg.data.use_mtranse_format)
logger.info(f"Data: {data.summary()}")
pd.DataFrame.from_dict(data.summary(), orient="index", columns=["valeur"])

### 3.1 A few aligned pairs (ground truth)

In [ ]:
rows = []
for e1, e2 in data.test_pairs[:8]:
    rows.append({"KG1 id": e1, "KG1 entity": data.ent_uri[e1].split("/")[-1],
                 "KG2 id": e2, "KG2 entity": data.ent_uri[e2].split("/")[-1]})
pd.DataFrame(rows)

### 3.2 AliNet graph construction
`build_alinet_graph` produces the normalised 1-hop adjacency (`A_hat = D^{-1/2}(A+I)D^{-1/2}`,
sparse) and the 2-hop edge list (capped at `max_two_hop` per entity, excluding 1-hop and self).

In [ ]:
adj1, two_hop = build_alinet_graph(data, max_two_hop=cfg.model.max_two_hop, seed=cfg.experiment.seed)
logger.info(f"Graph: adj1 nnz={adj1._nnz()} | 2-hop edges={two_hop.shape[1]}")

# 1-hop degree distribution (diagonal excluded)
deg = np.asarray(adj1.to_dense().bool().sum(1)).ravel() - 1 if data.num_entities < 2000 else None
fig, ax = plt.subplots(figsize=(7.5, 4))
e_dst = two_hop[0].numpy()
twoh_per = np.bincount(e_dst, minlength=data.num_entities)
ax.hist(np.clip(twoh_per, 0, cfg.model.max_two_hop), bins=40, color=CYCLE[1], alpha=0.9)
style_axes(ax, "2-hop neighbours (sampled) per entity", "#2-hop neighbours", "frequency")
plt.tight_layout(); plt.show()

---
## 4. The AliNet model

`encode = forward_all()` encodes **all** entities in one pass (full-batch GNN) and returns the
normalised JK representation. Components (identical to `code/src/models/alinet.py`):
- **gated multi-hop** layer: 1-hop GCN + attentional 2-hop (scatter-softmax, chunked
  aggregation for memory) combined by a **gate**; **linear propagation**.
- **alignment loss** (margin-ranking, epsilon-truncated hard negatives).
- **relation-aware (TransE) loss** `||z_h + r - z_t||` on the GNN representations: anchors all
  entities structurally (the decisive link).

In [ ]:
"""The AliNet model.

AliNet (Sun, Wang, Hu, Li, Zhang, Qu, Li - "Knowledge Graph Alignment Network
with Gated Multi-hop Neighborhood Aggregation", AAAI 2020) aligns entities with
a GNN that mitigates the *neighbourhood non-isomorphism* across KGs by combining:

1. **1-hop aggregation** - a GCN-style pass over the symmetrically-normalised
   adjacency ``A_hat`` (with self-loops):  ``g1 = A_hat (H W1)``.
2. **2-hop aggregation with attention** - distant neighbours are noisier, so they
   are aggregated with a GAT-style attention over the (capped) 2-hop edges:
   ``alpha_{ik} = softmax_k(LeakyReLU(a1.W2 h_i + a2.W2 h_k))``, ``g2 = sum_k alpha_{ik} W2 h_k``.
3. **Gating** - a learned gate combines the two:
   ``out = sigma(g1 W_g + b) * g1 + (1 - sigma(...)) * g2``.

The final entity representation is the **concatenation** of the input embedding
and every layer's output (JK-style), L2-normalised, and used directly for
alignment / evaluation (distance in embedding space).
"""



# --------------------------------------------------------------------------- #
#  Scatter helpers (avoid a torch_scatter dependency)
# --------------------------------------------------------------------------- #
def scatter_softmax(scores: torch.Tensor, index: torch.Tensor, n: int) -> torch.Tensor:
    """Softmax of ``scores`` grouped by ``index`` (each value in [0, n))."""
    mx = scores.new_full((n,), float("-inf")).index_reduce_(0, index, scores, "amax", include_self=True)
    mx = torch.nan_to_num(mx, neginf=0.0)
    s = (scores - mx[index]).exp()
    denom = torch.zeros(n, device=scores.device, dtype=scores.dtype).index_add_(0, index, s)
    return s / (denom[index] + 1e-16)


def scatter_add(src: torch.Tensor, index: torch.Tensor, n: int) -> torch.Tensor:
    """Sum rows of ``src`` into ``n`` buckets given by ``index``."""
    out = torch.zeros((n, src.shape[1]), device=src.device, dtype=src.dtype)
    return out.index_add_(0, index, src)


# --------------------------------------------------------------------------- #
#  Gated multi-hop layer
# --------------------------------------------------------------------------- #
class AliNetLayer(nn.Module):
    # Output is LINEAR (no ReLU): for entity alignment, a ReLU between propagation
    # layers destroys the structural signal (halves the features each layer) and
    # cripples generalisation - linear/spectral propagation works far better here.
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.0):
        super().__init__()
        self.W1 = nn.Linear(in_dim, out_dim, bias=False)   # 1-hop transform
        self.W2 = nn.Linear(in_dim, out_dim, bias=False)   # 2-hop transform
        self.a1 = nn.Parameter(torch.zeros(out_dim))       # attention (centre)
        self.a2 = nn.Parameter(torch.zeros(out_dim))       # attention (neighbour)
        self.gate = nn.Linear(out_dim, out_dim)            # gating on the 1-hop signal
        self.leaky = nn.LeakyReLU(0.2)
        self.dropout = dropout
        nn.init.xavier_uniform_(self.W1.weight)
        nn.init.xavier_uniform_(self.W2.weight)
        nn.init.xavier_uniform_(self.a1.view(1, -1)); nn.init.xavier_uniform_(self.a2.view(1, -1))

    def forward(self, h, adj1, e_dst, e_src):
        n = h.shape[0]
        # 1-hop GCN aggregation
        g1 = torch.sparse.mm(adj1, self.W1(h))                       # (N, out)
        # 2-hop attention aggregation (memory-efficient over the edge list)
        wh = self.W2(h)                                              # (N, out)
        if e_dst.numel() > 0:
            # attention scores via NODE-level projections (avoids materialising E x out)
            s1 = (wh * self.a1).sum(-1)                              # (N,)
            s2 = (wh * self.a2).sum(-1)                              # (N,)
            score = self.leaky(s1[e_dst] + s2[e_src])               # (E,)
            alpha = scatter_softmax(score, e_dst, n)                # (E,)
            if self.dropout > 0 and self.training:
                alpha = F.dropout(alpha, p=self.dropout)
            # weighted sum in edge-chunks so peak memory is (chunk x out), not (E x out)
            g2 = torch.zeros_like(g1)
            chunk = 500_000
            for s in range(0, e_src.shape[0], chunk):
                sl = slice(s, s + chunk)
                g2 = g2.index_add(0, e_dst[sl], alpha[sl].unsqueeze(-1) * wh[e_src[sl]])
        else:
            g2 = torch.zeros_like(g1)
        gate = torch.sigmoid(self.gate(g1))            # gating is the only non-linearity
        return gate * g1 + (1.0 - gate) * g2           # linear propagation (no ReLU)


# --------------------------------------------------------------------------- #
#  AliNet
# --------------------------------------------------------------------------- #
class AliNet(nn.Module):
    def __init__(self, num_entities, adj1, two_hop, embed_dim=300,
                 layer_dims=(300, 300), dropout=0.0, init="xavier",
                 normalize_embeddings=True, num_relations=0):
        super().__init__()
        self.normalize_embeddings = normalize_embeddings
        self.feat_dropout = dropout
        self.ent_emb = nn.Embedding(num_entities, embed_dim)
        # NB: xavier on a (num_entities, dim) table gives a tiny range and makes all
        # entities nearly identical (no discrimination => no generalisation). Init the
        # entity table with unit-scale normal vectors so they are spread on the sphere.
        nn.init.normal_(self.ent_emb.weight, std=1.0)
        with torch.no_grad():
            self.ent_emb.weight.data = F.normalize(self.ent_emb.weight.data, dim=-1)

        dims = [embed_dim] + list(layer_dims)
        self.layers = nn.ModuleList(
            [AliNetLayer(dims[i], dims[i + 1], dropout=dropout) for i in range(len(layer_dims))]
        )
        # graph structures are registered as buffers (move with .to(device), not trained)
        self.register_buffer("_adj1", adj1)
        self.register_buffer("e_dst", two_hop[0])
        self.register_buffer("e_src", two_hop[1])
        # JK = concat of the LAYER outputs only (NOT the raw input embedding).
        # Including the raw per-entity embedding lets the model memorise seeds via
        # their own identity and never generalise; the alignment signal must come
        # from neighbourhood aggregation, so we drop it from the representation.
        self.out_dim = sum(layer_dims)

        # relation embeddings live in the representation space (out_dim) for the
        # relation-aware (TransE-style) loss that anchors EVERY entity structurally.
        self.rel_emb = nn.Embedding(num_relations, self.out_dim) if num_relations else None
        if self.rel_emb is not None:
            nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward_all(self) -> torch.Tensor:
        """Compute the JK-concatenated (layer outputs) representation of ALL entities."""
        h = self.ent_emb.weight
        reps = []
        for layer in self.layers:
            if self.feat_dropout > 0 and self.training:
                h = F.dropout(h, p=self.feat_dropout)
            h = layer(h, self._adj1, self.e_dst, self.e_src)
            reps.append(h)
        z = torch.cat(reps, dim=-1)                    # (N, out_dim)
        return F.normalize(z, dim=-1) if self.normalize_embeddings else z


# --------------------------------------------------------------------------- #
#  Loss (limit-based contrastive on precomputed representations)
# --------------------------------------------------------------------------- #
def alinet_align_loss(z, e1, e2, neg_l, neg_r, margin):
    """**Margin-ranking** alignment loss on rows of a precomputed ``z`` (AliNet/GCN-Align).

    ``z`` is the full-graph representation (so the GNN runs once per step). All
    index tensors share the same length (``e1`` / ``e2`` are repeated ``neg`` times
    by the trainer to align with ``neg_r`` / ``neg_l``). A *relative* margin keeps a
    gradient flowing until each negative is at least ``margin`` farther than the
    gold pair - unlike a limit-based loss it does not saturate to zero after the
    few seed pairs are separated, so the GNN keeps learning generalisable structure.

        L = mean( [ margin + d(x,y) - d(x, y-) ]+ )   (+ symmetric left corruption)
    """
    d_pos = torch.norm(z[e1] - z[e2], p=2, dim=-1)
    d_neg_r = torch.norm(z[e1] - z[neg_r], p=2, dim=-1)     # corrupt right (KG2)
    d_neg_l = torch.norm(z[neg_l] - z[e2], p=2, dim=-1)     # corrupt left (KG1)
    loss_r = F.relu(margin + d_pos - d_neg_r).mean()
    loss_l = F.relu(margin + d_pos - d_neg_l).mean()
    return 0.5 * (loss_r + loss_l)


def alinet_relation_loss(z, rel_emb, pos, neg_t, margin):
    """Relation-aware (TransE-style) loss on the GNN representations ``z``.

    For a triple ``(h, r, t)`` the score is ``||z_h + r_r - z_t||`` with learnable
    relation vectors ``r_r``. A margin-ranking objective against tail-corrupted
    triples shapes ``z`` so that EVERY entity (not just the 11% seeds) gets a
    structural signal - the missing anchor that caps a pure entity-alignment GNN.
    This is AliNet's relation-aware component.
    """
    h = z[pos[:, 0]]
    r = rel_emb(pos[:, 1])
    t = z[pos[:, 2]]
    pos_s = torch.norm(h + r - t, p=2, dim=-1)
    neg_s = torch.norm(h + r - z[neg_t], p=2, dim=-1)
    return F.relu(margin + pos_s - neg_s).mean()


def alinet_limit_loss(z, e1, e2, neg_l, neg_r, pos_margin, neg_margin, neg_weight=1.0):
    """**Limit-based** (absolute-margin) alignment loss on rows of ``z``.

    Pulls aligned pairs below ``pos_margin`` and pushes negatives above
    ``neg_margin``. Unlike margin-ranking, the negative push is **bounded** (it
    stops once a negative is far enough), so it does not over-scatter the tail
    when combined with hard (epsilon-truncated) negatives - which is what wrecks the
    ranking loss here. Same recipe that worked for BootEA.
    """
    d_pos = torch.norm(z[e1] - z[e2], p=2, dim=-1)
    d_neg_r = torch.norm(z[e1] - z[neg_r], p=2, dim=-1)
    d_neg_l = torch.norm(z[neg_l] - z[e2], p=2, dim=-1)
    l_pos = F.relu(d_pos - pos_margin).mean()
    l_neg = 0.5 * (F.relu(neg_margin - d_neg_r).mean() + F.relu(neg_margin - d_neg_l).mean())
    return l_pos + neg_weight * l_neg

In [ ]:
set_seed(cfg.experiment.seed)
model = AliNet(
    num_entities=data.num_entities, adj1=adj1.to(device), two_hop=two_hop.to(device),
    embed_dim=cfg.model.embed_dim, layer_dims=tuple(cfg.model.layer_dims),
    dropout=cfg.model.get("dropout", 0.0), normalize_embeddings=cfg.model.normalize_embeddings,
    num_relations=data.num_relations,
).to(device)
logger.info(f"AliNet : {sum(p.numel() for p in model.parameters())/1e6:.2f}M parametres | "
            f"rep_dim={model.out_dim}")

---
## 5. Metrics: MRR, Hit@k, CSLS (identical to `code/src/utils/metrics.py`)

In [ ]:
"""Entity-alignment evaluation: MRR and Hit@k, with cosine or CSLS scoring.

Protocol (standard for DBP15K): the test set is a list of gold pairs
``(e1_i, e2_i)``. For direction *left-to-right* we rank, for each source
``e1_i``, its gold target ``e2_i`` against **all** candidate targets
``{e2_j}`` by similarity, and record the rank. Hit@k = fraction of sources
whose gold rank <= k; MRR = mean of ``1 / rank``.

CSLS (Cross-domain Similarity Local Scaling, Lample et al. 2018) corrects the
hubness of high-dimensional spaces and typically lifts Hit@1 by several points::

    csls(x, y) = 2.cos(x, y) - r_T(x) - r_S(y)

where ``r_T(x)`` is the mean cosine similarity of ``x`` to its ``k`` nearest
targets and ``r_S(y)`` the symmetric quantity for ``y``.
"""



def _mean_topk_sim(sim: torch.Tensor, k: int, dim: int) -> torch.Tensor:
    """Mean of the top-``k`` similarities along ``dim`` (CSLS local scaling)."""
    k = min(k, sim.shape[dim])
    vals, _ = sim.topk(k, dim=dim)
    return vals.mean(dim=dim)


@torch.no_grad()
def _rank_metrics(sim: torch.Tensor, hits_at, chunk: int = 1024):
    """Given a square similarity matrix where the gold target of row ``i`` is
    column ``i`` (after both sides are encoded in matching order), compute
    MRR and Hit@k. Done in row-chunks to bound memory."""
    n = sim.shape[0]
    device = sim.device
    ranks = torch.empty(n, device=device)
    gold = torch.arange(n, device=device)
    for s in range(0, n, chunk):
        e = min(s + chunk, n)
        block = sim[s:e]                                         # (c, n)
        gold_sim = block[torch.arange(e - s, device=device), gold[s:e]].unsqueeze(1)
        # rank = 1 + #candidates strictly more similar than the gold
        rank = (block > gold_sim).sum(1) + 1
        ranks[s:e] = rank.float()
    out = {"MRR": (1.0 / ranks).mean().item()}
    for k in hits_at:
        out[f"Hit@{k}"] = (ranks <= k).float().mean().item()
    out["MeanRank"] = ranks.mean().item()
    return out


@torch.no_grad()
def evaluate_alignment(
    z_left: torch.Tensor,
    z_right: torch.Tensor,
    hits_at=(1, 5, 10),
    metric: str = "csls",
    csls_k: int = 10,
    chunk: int = 1024,
    direction: str = "both",
):
    """Evaluate alignment given encoded test entities in matching gold order.

    ``z_left[i]`` aligns to ``z_right[i]``. Embeddings are L2-normalised here so
    cosine == dot product. Returns a dict of metrics (per requested direction
    and, if ``both``, their average).
    """
    zl = torch.nn.functional.normalize(z_left, dim=-1)
    zr = torch.nn.functional.normalize(z_right, dim=-1)
    sim = zl @ zr.t()                                            # (n, n) cosine

    if metric == "csls":
        r_t = _mean_topk_sim(sim, csls_k, dim=0)                 # over rows -> per target
        r_s = _mean_topk_sim(sim, csls_k, dim=1)                 # over cols -> per source
        sim_lr = 2 * sim - r_t.unsqueeze(0) - r_s.unsqueeze(1)
        sim_rl = sim_lr.t()
    elif metric in ("cosine", "l2"):
        sim_lr = sim
        sim_rl = sim.t()
    else:
        raise ValueError(f"unknown metric {metric!r}")

    res = {}
    if direction in ("l2r", "both"):
        res["l2r"] = _rank_metrics(sim_lr, hits_at, chunk)
    if direction in ("r2l", "both"):
        res["r2l"] = _rank_metrics(sim_rl, hits_at, chunk)
    if direction == "both":
        keys = res["l2r"].keys()
        res["avg"] = {k: 0.5 * (res["l2r"][k] + res["r2l"][k]) for k in keys}
    return res


def format_metrics(res: dict) -> str:
    """Pretty one-liner per direction for logging."""
    lines = []
    for d, m in res.items():
        parts = [f"MRR={m['MRR']:.4f}"]
        parts += [f"{k}={v:.4f}" for k, v in m.items() if k.startswith("Hit@")]
        parts.append(f"MR={m['MeanRank']:.1f}")
        lines.append(f"[{d:>3}] " + " ".join(parts))
    return " | ".join(lines)

In [ ]:
test_left  = torch.from_numpy(data.test_pairs[:, 0]).to(device)
test_right = torch.from_numpy(data.test_pairs[:, 1]).to(device)
with torch.no_grad():
    z = model.forward_all()
res0 = evaluate_alignment(z[test_left], z[test_right], hits_at=tuple(cfg.eval.hits_at),
                          metric=cfg.eval.metric, csls_k=cfg.eval.csls_k,
                          chunk=cfg.eval.eval_chunk, direction=cfg.eval.direction)
print("Baseline (UNtrained model):")
print(format_metrics(res0))

---
## 6. Training (AliNet, full-batch)

The `AliNetTrainer` (identical to the class of the same name in `code/src/trainer.py`): each
epoch it encodes the whole graph once, applies the alignment loss (mixed hard negatives) plus
the relation-aware loss, evaluates periodically (CSLS), logs everything and applies early
stopping. `tqdm` bars in the terminal.

In [ ]:
class AliNetTrainer:
    """AliNet training loop (Sun et al., AAAI 2020).

    AliNet is a **full-batch** GNN: every epoch the gated multi-hop network
    encodes *all* entities once (``model.forward_all()``), then a limit-based
    contrastive alignment loss is applied to all seed pairs with **epsilon-truncated**
    (nearest cross-KG) negatives. No TransE triples, no swapping; bootstrapping
    is optional (off by default, matching the base paper).
    """

    def __init__(self, cfg, data: DBP15K, model: AliNet, run_dir: Path, logger):
        self.cfg = cfg
        self.data = data
        self.model = model
        self.run_dir = Path(run_dir)
        self.log = logger
        self.device = next(model.parameters()).device

        self.kg1_ents = torch.from_numpy(data.kg1_ents).to(self.device)
        self.kg2_ents = torch.from_numpy(data.kg2_ents).to(self.device)
        self.seed_l = torch.from_numpy(data.train_pairs[:, 0]).to(self.device)
        self.seed_r = torch.from_numpy(data.train_pairs[:, 1]).to(self.device)
        self.hard_r = None          # (S, C) nearest KG2 to each seed_l
        self.hard_l = None          # (S, C) nearest KG1 to each seed_r
        self.pseudo = None          # optional bootstrapped pairs (S2, 2)

        # triples + per-KG entity pools for the relation-aware (TransE) loss
        self.kg_of_ent = torch.as_tensor(kg_of_entity(data), device=self.device)
        self.triples = torch.from_numpy(np.concatenate([data.triples1, data.triples2], 0)).to(self.device)

        params = model.parameters()
        if cfg.train.optimizer.lower() == "adam":
            self.optimizer = torch.optim.Adam(params, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        else:
            self.optimizer = torch.optim.SGD(params, lr=cfg.train.lr, weight_decay=cfg.train.weight_decay)
        self.scheduler = None
        if str(cfg.train.get("lr_schedule", "none")).lower() == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=cfg.train.epochs, eta_min=cfg.train.lr * 0.02)

        self.test_left = torch.from_numpy(data.test_pairs[:, 0]).to(self.device)
        self.test_right = torch.from_numpy(data.test_pairs[:, 1]).to(self.device)

        self.loss_hist, self.metric_hist = [], []
        self.best_mrr, self.best_epoch, self.no_improve = -1.0, -1, 0

    # ------------------------------------------------------------------ #
    #  epsilon-truncated nearest cross-KG negatives for the seed (and pseudo) pairs
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def refresh_hard_negatives(self):
        C = self.cfg.train.hard_negatives.num_candidates
        self.model.eval()
        z = self.model.forward_all()
        left = self.seed_l if self.pseudo is None else torch.cat([self.seed_l, self.pseudo[:, 0]])
        right = self.seed_r if self.pseudo is None else torch.cat([self.seed_r, self.pseudo[:, 1]])
        z_kg2 = z[self.kg2_ents]
        z_kg1 = z[self.kg1_ents]

        def nearest(zq, zpool, pool_ids):
            out = torch.empty((zq.shape[0], C), dtype=torch.long, device=self.device)
            for s in range(0, zq.shape[0], 2048):
                top = (zq[s:s + 2048] @ zpool.t()).topk(C, dim=1).indices
                out[s:s + 2048] = pool_ids[top]
            return out

        self.hard_r = nearest(z[left], z_kg2, self.kg2_ents)
        self.hard_l = nearest(z[right], z_kg1, self.kg1_ents)

    def _sample_neg(self, table, gold, pool):
        """One random candidate per row of ``table`` (avoid the gold target)."""
        sel = torch.randint(table.shape[1], (table.shape[0], 1), device=self.device)
        out = torch.gather(table, 1, sel).squeeze(1)
        clash = out == gold
        if clash.any():
            out[clash] = pool[torch.randint(len(pool), (int(clash.sum()),), device=self.device)]
        return out

    # ------------------------------------------------------------------ #
    #  One full-batch training epoch
    # ------------------------------------------------------------------ #
    def train_epoch(self, epoch: int):
        self.model.train()
        c = self.cfg.train
        z = self.model.forward_all()

        left = self.seed_l if self.pseudo is None else torch.cat([self.seed_l, self.pseudo[:, 0]])
        right = self.seed_r if self.pseudo is None else torch.cat([self.seed_r, self.pseudo[:, 1]])
        n = c.neg_samples
        e1 = left.repeat_interleave(n)
        e2 = right.repeat_interleave(n)
        B = e1.shape[0]
        rand_r = self.kg2_ents[torch.randint(len(self.kg2_ents), (B,), device=self.device)]
        rand_l = self.kg1_ents[torch.randint(len(self.kg1_ents), (B,), device=self.device)]
        if self.hard_r is not None:
            hard_r = self._sample_neg(self.hard_r.repeat_interleave(n, 0), e2, self.kg2_ents)
            hard_l = self._sample_neg(self.hard_l.repeat_interleave(n, 0), e1, self.kg1_ents)
            strat = str(c.hard_negatives.get("strategy", "hard")).lower()
            if strat == "mixed":
                # gently mix random + hard negatives (hard alone tends to scatter the tail)
                ratio = c.hard_negatives.get("hard_ratio", 0.5)
                use_hard = torch.rand(B, device=self.device) < ratio
                neg_r = torch.where(use_hard, hard_r, rand_r)
                neg_l = torch.where(use_hard, hard_l, rand_l)
            else:
                neg_r, neg_l = hard_r, hard_l
        else:
            neg_r, neg_l = rand_r, rand_l

        if str(c.get("loss", "ranking")).lower() == "limit":
            align = alinet_limit_loss(z, e1, e2, neg_l, neg_r,
                                      c.align_pos_margin, c.align_neg_margin, c.get("align_neg_weight", 1.0))
        else:
            align = alinet_align_loss(z, e1, e2, neg_l, neg_r, c.align_margin)

        # relation-aware (TransE) loss on a sampled batch of triples -> anchors all entities
        rel_val = 0.0
        rcfg = c.get("relation", None)
        if rcfg and rcfg.enabled and self.model.rel_emb is not None:
            M_ = self.triples.shape[0]
            bs = min(rcfg.batch_size, M_)
            idx = torch.randint(M_, (bs,), device=self.device)
            pos = self.triples[idx]
            # corrupt the tail with a random entity from the SAME KG as the true tail
            kg_t = self.kg_of_ent[pos[:, 2]]
            rt2 = self.kg2_ents[torch.randint(len(self.kg2_ents), (bs,), device=self.device)]
            rt1 = self.kg1_ents[torch.randint(len(self.kg1_ents), (bs,), device=self.device)]
            neg_t = torch.where(kg_t == 0, rt1, rt2)
            rel = alinet_relation_loss(z, self.model.rel_emb, pos, neg_t, rcfg.margin)
            rel_val = rel.item()
            loss = align + rcfg.weight * rel
        else:
            loss = align

        self.optimizer.zero_grad()
        loss.backward()
        if c.grad_clip and c.grad_clip > 0:
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), c.grad_clip)
        self.optimizer.step()
        return {"loss": loss.item(), "align": align.item(), "rel": rel_val}

    # ------------------------------------------------------------------ #
    #  Optional bootstrapping (editable MWGM) -> extra labelled pairs
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def bootstrap(self):
        bs = self.cfg.train.bootstrap
        self.model.eval()
        z = self.model.forward_all()
        zl = z[self.test_left]; zr = z[self.test_right]
        cos = zl @ zr.t()
        k = self.cfg.eval.csls_k
        r_t = cos.topk(min(k, cos.shape[0]), dim=0).values.mean(0)
        r_s = cos.topk(min(k, cos.shape[1]), dim=1).values.mean(1)
        csls = 2 * cos - r_t.unsqueeze(0) - r_s.unsqueeze(1)
        idx = torch.arange(len(self.test_left), device=self.device)
        best_r = csls.argmax(1); best_l = csls.argmax(0)
        keep = (best_l[best_r] == idx) & (cos[idx, best_r] >= bs.threshold)
        cand = idx[keep]
        if len(cand) == 0:
            self.pseudo = None
            return 0
        order = torch.argsort(cos[cand, best_r[cand]], descending=True)
        cand = cand[order][: bs.max_add]
        used, pairs = set(), []
        for li, ri in zip(cand.tolist(), best_r[cand].tolist()):
            if ri in used:
                continue
            used.add(ri); pairs.append((int(self.test_left[li]), int(self.test_right[ri])))
        self.pseudo = torch.tensor(pairs, dtype=torch.long, device=self.device)
        return len(pairs)

    # ------------------------------------------------------------------ #
    #  Evaluation / persistence / plots
    # ------------------------------------------------------------------ #
    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        ec = self.cfg.eval
        z = self.model.forward_all()
        return evaluate_alignment(z[self.test_left], z[self.test_right],
                                    hits_at=tuple(ec.hits_at), metric=ec.metric,
                                    csls_k=ec.csls_k, chunk=ec.eval_chunk, direction=ec.direction)

    def save_checkpoint(self, name, epoch, res=None):
        torch.save({"epoch": epoch, "model_state": self.model.state_dict(),
                    "config": self.cfg.to_plain(), "metrics": res}, self.run_dir / name)

    def save_embeddings(self):
        with torch.no_grad():
            z = self.model.forward_all().detach().cpu()
        torch.save({"entity_repr": z, "ent_emb": self.model.ent_emb.weight.detach().cpu()},
                   self.run_dir / self.cfg.logging.embeddings_name)

    def _append_csv(self, name, row, header_order=None):
        path = self.run_dir / name
        new = not path.exists()
        with open(path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=header_order or list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)

    def plot_curves(self):
        set_modern_dark_style()
        if self.loss_hist:
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_loss_curves(self.loss_hist, ax=ax, keys=("loss",))
            fig.tight_layout(); fig.savefig(self.run_dir / self.cfg.logging.plots.loss_curve); plt.close(fig)
        if self.metric_hist:
            fig, ax = plt.subplots(figsize=(8, 5))
            plot_metric_curves(self.metric_hist, ax=ax)
            fig.tight_layout(); fig.savefig(self.run_dir / self.cfg.logging.plots.metrics_curve); plt.close(fig)

    # ------------------------------------------------------------------ #
    #  Main loop
    # ------------------------------------------------------------------ #
    def fit(self):
        cfg = self.cfg
        self.log.info(f"Run directory: {self.run_dir}")
        self.log.info(f"Device: {self.device} | entities={self.data.num_entities} "
                      f"2hop_edges={self.model.e_dst.numel()} rep_dim={self.model.out_dim} "
                      f"seed_pairs={len(self.seed_l)} test_pairs={len(self.test_left)}")
        boot, hn = cfg.train.bootstrap, cfg.train.hard_negatives
        t0 = time.time()

        for epoch in tqdm(range(1, cfg.train.epochs + 1), desc="AliNet", ncols=100):
            losses = self.train_epoch(epoch)
            losses["epoch"] = epoch
            self.loss_hist.append(losses)
            self._append_csv(cfg.logging.loss_csv, losses, ["epoch", "loss", "align", "rel"])
            msg = f"epoch {epoch:>4}/{cfg.train.epochs} | loss={losses['loss']:.4f}"

            if hn.enabled and epoch >= hn.start_epoch and (epoch - hn.start_epoch) % hn.refresh_every == 0:
                self.refresh_hard_negatives()
                msg += " | hard-neg refreshed"
            if boot.enabled and epoch >= boot.start_epoch and (epoch - boot.start_epoch) % boot.every == 0:
                added = self.bootstrap()
                msg += f" | bootstrap: {added} pairs"
            if self.scheduler is not None:
                self.scheduler.step()

            if epoch % cfg.eval.every == 0 or epoch == cfg.train.epochs:
                self.log.info(msg)
                res = self.evaluate()
                self.log.info("           " + format_metrics(res))
                ref = res.get("avg", res.get("l2r"))
                self._append_csv(cfg.logging.metrics_csv, {"epoch": epoch, **{k: ref[k] for k in ref}})
                self.metric_hist.append({"epoch": epoch, "MRR": ref["MRR"],
                                         **{k: v for k, v in ref.items() if k.startswith("Hit@")}})
                self.plot_curves()
                if ref["MRR"] > self.best_mrr:
                    self.best_mrr, self.best_epoch, self.no_improve = ref["MRR"], epoch, 0
                    if cfg.logging.save_best:
                        self.save_checkpoint("model_best.pt", epoch, res)
                        self.log.info(f"           -> new best MRR={self.best_mrr:.4f} (saved model_best.pt)")
                else:
                    self.no_improve += 1
                patience = cfg.train.get("early_stop_patience", 0)
                if patience and self.no_improve >= patience:
                    self.log.info(f"           early stop: no MRR improvement for {patience} evals "
                                  f"(best={self.best_mrr:.4f} @ {self.best_epoch}).")
                    break

        if cfg.logging.save_last:
            self.save_checkpoint(cfg.logging.checkpoint_name, cfg.train.epochs)
        self.save_embeddings()
        self.plot_curves()
        self.log.info(f"Done in {(time.time()-t0)/60:.1f} min. Best MRR={self.best_mrr:.4f} @ epoch {self.best_epoch}.")
        return {"best_mrr": self.best_mrr, "best_epoch": self.best_epoch,
                "metric_hist": self.metric_hist, "loss_hist": self.loss_hist}

In [ ]:
set_seed(cfg.experiment.seed)
trainer = AliNetTrainer(cfg, data, model, run_dir, logger)
history = trainer.fit()

---
## 7. Curves and results

In [ ]:
loss_df = pd.read_csv(run_dir / cfg.logging.loss_csv)
met_df  = pd.read_csv(run_dir / cfg.logging.metrics_csv)
display(met_df.tail(10))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_loss_curves(history["loss_hist"], ax=axes[0], keys=("loss", "align", "rel"))
plot_metric_curves(history["metric_hist"], ax=axes[1])
plt.tight_layout(); plt.show()
print(f"Best MRR = {history['best_mrr']:.4f} at epoch {history['best_epoch']}")

---
## 8. Final evaluation and qualitative analysis

In [ ]:
ckpt = torch.load(run_dir / "model_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state"])
logger.info(f"Best checkpoint reloaded (epoch {ckpt['epoch']}).")
with torch.no_grad():
    z = model.forward_all()
res = evaluate_alignment(z[test_left], z[test_right], hits_at=tuple(cfg.eval.hits_at),
                         metric=cfg.eval.metric, csls_k=cfg.eval.csls_k,
                         chunk=cfg.eval.eval_chunk, direction="both")
print(format_metrics(res))

In [ ]:
with torch.no_grad():
    zl = F.normalize(z[test_left], dim=-1); zr = F.normalize(z[test_right], dim=-1)
    top = (zl @ zr.t()).topk(3, dim=1).indices.cpu().numpy()
rows = []
for i in range(10):
    status = "hit@1" if top[i][0] == i else ("top-3" if i in top[i] else "miss")
    rows.append({"source (KG1)": data.ent_uri[int(test_left[i])].split("/")[-1],
                 "or (KG2)": data.ent_uri[int(test_right[i])].split("/")[-1],
                 "top-1": data.ent_uri[int(test_right[top[i][0]])].split("/")[-1],
                 "top-2": data.ent_uri[int(test_right[top[i][1]])].split("/")[-1],
                 "top-3": data.ent_uri[int(test_right[top[i][2]])].split("/")[-1],
                 "statut": status})
pd.DataFrame(rows)

---
## 9. Saved artefacts

In [ ]:
print("Run dir :", run_dir, "\n")
for p in sorted(run_dir.iterdir()):
    print(f"  {p.name:<22} {p.stat().st_size/1024:8.1f} KB")

---
## 10. Comparison with the paper

In [ ]:
res_avg = res.get("avg", res.get("l2r"))
summary_df = pd.DataFrame([
    {"model": "AliNet (paper)", "Hit@1": 0.539, "Hit@10": 0.826, "MRR": 0.628},
    {"model": "This notebook",     "Hit@1": res_avg["Hit@1"], "Hit@10": res_avg["Hit@10"], "MRR": res_avg["MRR"]},
]).set_index("model")
display(summary_df.round(4))

fig, ax = plt.subplots(figsize=(8, 4.5))
names = ["Hit@1", "Hit@10", "MRR"]; x = np.arange(len(names)); w = 0.38
ax.bar(x - w/2, [0.539, 0.826, 0.628], w, label="AliNet (paper)", color=CYCLE[5])
ax.bar(x + w/2, [res_avg["Hit@1"], res_avg["Hit@10"], res_avg["MRR"]], w, label="This notebook", color=CYCLE[0])
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 1)
style_axes(ax, f"AliNet paper vs notebook ({cfg.data.lang})", None, "score"); ax.legend()
plt.tight_layout(); plt.show()

---
## 11. Notes: what makes AliNet work (debugging lessons)

- **LINEAR propagation** (no ReLU between layers): a ReLU GCN caps at ~0.20 Hit@1, the linear
  version reaches ~0.40. The ReLU destroys the structural signal.
- **Unit init** of the embeddings (xavier on an (N, dim) table gives tiny values => all
  entities identical => no generalisation).
- **JK = layer outputs only** (NOT the raw embedding): otherwise the model memorises the seeds
  via their own identity (seedHit1=1.0, testHit1~0).
- **Relation-aware (TransE) loss** = the decisive link: it anchors ALL entities (MeanRank
  200 -> 75). Without it, we plateau at the GCN-Align level (~0.45/0.75).
- **Hard epsilon-truncated negatives**: harmful WITHOUT the relation anchor (they scatter the
  tail); beneficial WITH the anchor (Hit@1 0.45 -> 0.53).
- **2 layers** (3+ over-smooths); 2-hop attention **chunked** for memory; **CSLS** at eval.

**References**
- Sun et al., AliNet, AAAI 2020 - https://arxiv.org/pdf/1911.08936v1
- Wang et al., GCN-Align, EMNLP 2018.
- Lample et al., Word Translation Without Parallel Data (CSLS), ICLR 2018.